# Purchase-model training and dynamic pricing precomputation

Train an XGBoost model with monotonic price constraints to estimate purchase probability from customer covariates and offered price.
- **Economic Monotonicity**: Enforces that purchase probability is non-increasing in offered price ((p, x) \le d(p', x)$ for  \ge p'$).
- **Evaluation**: Evaluated on held-out purchase decisions using MSE, Log-Loss, and ROC AUC.
- **Bellman Equation Precomputation**: Computes the 2D Dynamic Programming table of marginal opportunity costs for capacity-constrained pricing (Part 1 Step 3).
- **Saved Artifacts**: Saves the model to  and the DP opportunity cost table to .


In [1]:
from pathlib import Path
import os
import sys
import time
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, log_loss, roc_auc_score

# Resolve paths once so every notebook uses the project folder.
project_folder = next((
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / "project_code_2026_publicshare",
                   parent / "project_2026/project_code_2026_publicshare",
                   parent / "private_github/project_2026/project_code_2026_publicshare")
    if (folder / "pricing_helpers.py").is_file()
), None)
if project_folder is None:
    raise FileNotFoundError("Open the project folder as your notebook workspace.")
os.chdir(project_folder)
sys.path.insert(0, str(project_folder))

data = pd.read_csv(project_folder / "data/train_prices_decisions_2026.csv")
covariates = ["Covariate1", "Covariate2", "Covariate3"]
features = covariates + ["price_item"]


In [2]:
# 1. Validate on 80/20 train/test split
train, test = train_test_split(data, test_size=0.2, random_state=42)
eval_model = xgb.XGBRegressor(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.03,
    objective="binary:logistic",
    monotone_constraints=(0, 0, 0, -1),
    random_state=42
)
eval_model.fit(train[features], train.item_bought.astype(float))

held_out_probs = np.clip(eval_model.predict(test[features]), 0, 1)
print(f"Held-out Mean Squared Error: {mean_squared_error(test.item_bought, held_out_probs):.5f}")
print(f"Held-out Log-Loss: {log_loss(test.item_bought, held_out_probs):.5f}")
print(f"Held-out ROC AUC: {roc_auc_score(test.item_bought, held_out_probs):.5f}")


Held-out Mean Squared Error: 0.03574
Held-out Log-Loss: 0.11964
Held-out ROC AUC: 0.99145


## Train Final Model, Save, and Verify Reload

Fit the final model on all 50,000 observations and save to .

In [3]:
# Fit final model on all data
model = xgb.XGBRegressor(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.03,
    objective="binary:logistic",
    monotone_constraints=(0, 0, 0, -1),
    random_state=42
)
model.fit(data[features], data.item_bought.astype(float))

model_path = project_folder / "agents/helper_files_here/purchase_model.pkl"
with model_path.open("wb") as file:
    pickle.dump(model, file)

with model_path.open("rb") as file:
    loaded_model = pickle.load(file)

np.testing.assert_allclose(loaded_model.predict(test[features]), model.predict(test[features]))
print("Saved and verified reloaded model:", model_path)


Saved and verified reloaded model: /Users/aakarshgundu/Documents/Cornell Tech/ORIE 5355/Project/agents/helper_files_here/purchase_model.pkl


## Precompute Bellman Dynamic Program Table for Capacity Constraints

Precompute the 2D Bellman value function (T_{rem}, k)$ and marginal opportunity cost table $\Delta V(T_{rem}-1, k) = V(T_{rem}-1, k) - V(T_{rem}-1, k-1)$ over the customer population distribution for {rem} \in \{1, \dots, 20\}$ and  \in \{1, \dots, 20\}$.

In [4]:
t0 = time.time()
sample_cust = data[covariates].sample(n=5000, random_state=42).reset_index(drop=True)
price_grid = np.linspace(0, 350, 176)

repeated_covs = np.repeat(sample_cust.values, len(price_grid), axis=0)
tiled_prices = np.tile(price_grid, len(sample_cust))
X_eval = pd.DataFrame(repeated_covs, columns=covariates)
X_eval["price_item"] = tiled_prices
probs = np.clip(model.predict(X_eval), 0, 1).reshape((len(sample_cust), len(price_grid)))

V = np.zeros((21, 21))
for t_rem in range(1, 21):
    for k in range(1, 21):
        opp_cost = V[t_rem - 1, k] - V[t_rem - 1, k - 1]
        surplus = probs * (price_grid - opp_cost)
        max_surplus = np.maximum(0, np.max(surplus, axis=1))
        V[t_rem, k] = V[t_rem - 1, k] + np.mean(max_surplus)

opp_cost_table = np.zeros((21, 21))
for t_rem in range(1, 21):
    for k in range(1, 21):
        opp_cost_table[t_rem, k] = max(0.0, V[t_rem - 1, k] - V[t_rem - 1, k - 1])

opp_cost_path = project_folder / "agents/helper_files_here/opp_cost_table.npy"
np.save(opp_cost_path, opp_cost_table)
print("Saved DP opportunity cost table to:", opp_cost_path)
print(f"DP computation completed in {time.time() - t0:.2f}s")


Saved DP opportunity cost table to: /Users/aakarshgundu/Documents/Cornell Tech/ORIE 5355/Project/agents/helper_files_here/opp_cost_table.npy
DP computation completed in 0.54s


In [5]:
example = test[features].iloc[:5].copy()
probability = np.clip(loaded_model.predict(example), 0, 1)
example.assign(predicted_purchase_probability=probability,
               predicted_revenue=example.price_item * probability)


,Covariate1,Covariate2,Covariate3,price_item,predicted_purchase_probability,predicted_revenue
33553,1.0,8.632353,10.320575,110.177547,0.007037,0.775319
9427,1.0,0.948613,4.852005,189.172213,0.005956,1.126716
199,0.0,1.232130,6.285150,32.712328,0.993077,32.485850
12447,1.0,0.413627,7.551069,56.801858,0.551513,31.326957
39489,0.0,5.543664,8.695999,13.402927,0.999470,13.395818
